In [ ]:
from pathlib import Path
from IPython.display import Image, SVG, display
import getpass
import os
import re
import time

# Works when this notebook starts in notebooks/, the repository root, or Gadi.
candidates = [Path.cwd(), *Path.cwd().parents,
              Path(os.path.expandvars("/scratch/vp91/$USER/intro-to-cuda"))]
repo_root = next((path for path in candidates
                  if (path / "exercises/CMakeLists.txt").is_file()
                  and (path / "job-script").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the intro-to-cuda checkout.")
os.chdir(repo_root)
job_dir = repo_root / "job-script"
username = getpass.getuser()


def show_job_output(job_name, job_id):
    job_number = job_id.split(".")[0]
    log = job_dir / f"{job_name}.o{job_number}"
    if log.is_file():
        text = log.read_text()
        print(text, end="")
        return text
    else:
        print(f"Output is not available yet: {log.name}. Check qstat, then rerun this cell.")


def outputs_ready(paths, submitted_at):
    if all(path.is_file() and path.stat().st_mtime >= submitted_at for path in paths):
        return True
    print("New results are not available yet. Wait for this job to finish, then rerun this cell.")
    return False

print(f"Repository: {repo_root}")

def show_sample_output(filename, label="Saved sample output"):
    log = repo_root / "sample-outputs" / filename
    text = log.read_text()
    print(f"{label} ({log.name})\n")
    print(text, end="")
    return text

timing_log = None


# Exercise 01: Performance Basics

**Time: 20 min**

Compare matrix multiplication on the CPU with Intel MKL and on the GPU with
cuBLAS. Both supplied programs use double precision and compute `C = A × B + C`.
[Exercise instructions](../docs/source/tutorial/exercises/01-performance-basics.rst).

Run the setup cell first. Build the exercises once in
[02-gpu.ipynb](02-gpu.ipynb) before submitting jobs from this notebook.
After editing CUDA source, return to that notebook to rebuild.

Submission cells use the same `qsub` commands as the exercise documentation,
from `job-script`, and immediately show `qstat`. A submitted job may still be queued.
Use the monitoring cell to check again; rerunning a submission cell creates another job.
Read the PBS output and inspect generated files after the job finishes.

## Inspect the programs

Read the matrix size and repetition count in each source. Before comparing timings,
check what initialization, transfers and computation each timer includes.

<a href="../exercises/01-performance-basics/perf-gemm-intelmkl.c" target="_blank" rel="noopener noreferrer">Open perf-gemm-intelmkl.c</a>

<a href="../exercises/01-performance-basics/perf-gemm-cuda.cu" target="_blank" rel="noopener noreferrer">Open perf-gemm-cuda.cu</a>

## Submit the benchmark job

The job runs the CPU benchmark first and the GPU benchmark second. Compilation
happens only in `02-gpu.ipynb`; no temporary benchmark copies are built here.

In [ ]:
# Submit a new job, then inspect the queue.
timing_log = None
submitted_at = time.time()
submission = !cd "{job_dir}" && qsub 01-performance-basics.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")


### Check the queue again

Run this cell to monitor jobs without submitting another one. Once your job is no
longer listed, rerun its output/visualisation cells. Inspect the PBS log for errors.

In [ ]:
!qstat -u "{username}"


### Read the benchmark output

In [ ]:
!qstat -u "{username}"
timing_log = show_job_output("01-performance-basics", job_id)


### Use saved output while the job is queued

After running the setup cell, use this option to read the saved PBS log while the job is queued. The comparison below uses the selected log.

In [ ]:
timing_log = show_sample_output("01-performance-basics.o184402016")

## Compare

Run the comparison using the live log or the saved sample. CPU times measure the MKL call;
GPU times use events around the cuBLAS call. Initial GPU copies are outside the
GPU interval, and the GPU example does not copy results back for verification.
These are computation intervals, not full application runtimes.

In [ ]:
from statistics import median

log = timing_log
if log is None:
    log = show_sample_output("01-performance-basics.o184402016")
cpu_times_ms = []
gpu_times_ms = []
for value, period in re.findall(r"^Run \d+ completed in (\d+\.\d+) ms(\.?)$", log, re.MULTILINE):
    # The cuBLAS program prints a period after "ms"; the MKL program omits it.
    (gpu_times_ms if period else cpu_times_ms).append(float(value))

if cpu_times_ms and gpu_times_ms:
    print(f"Median CPU: {median(cpu_times_ms):.3f} ms")
    print(f"Median GPU: {median(gpu_times_ms):.3f} ms")
    print(f"CPU / GPU: {median(cpu_times_ms) / median(gpu_times_ms):.2f}")
else:
    print("The log is missing CPU or GPU timing rows.")


1. Explain which costs each measurement includes before comparing CPU and GPU.
2. Edit the source matrix sizes/repetitions if you want another experiment, rebuild
   in `02-gpu.ipynb`, and resubmit this same job.
3. How would including allocation and transfers change the comparison?